Install Libraries

In [ ]:
# !pip install -q --upgrade transformers datasets accelerate peft trl bitsandbytes

In [ ]:
# !pip install --upgrade torchao

Register the Model device

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

# padding is set so batches are rectangular

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)

In [ ]:
total_parameters = sum(p.numel() for p in model.parameters())
total_parameters

# Test the current Model with baseline inputs

In [ ]:
def test_generation(prompt):
  messages = [
      {
          "role": "user", "content": prompt
      }
  ]
  formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

  with torch.no_grad():
    output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
  generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
  return tokenizer.decode(generated_ids, skip_special_tokens=True)

In [ ]:
prompt_1 = "Write a short 4-line poem in hinglish about mountains."
prompt_2 = "Write a short 4-line poem about longing and quiet nights."

print("--- BASELINE OUTPUT 1 ---")
print(test_generation(prompt_1))
print("\n--- BASELINE OUTPUT 2 ---")
print(test_generation(prompt_2))

# Dataset Ingesion and ETL Pipeline

In [ ]:
from datasets import load_dataset

In [ ]:
# print("Downloading dataset from Hugging Face...")
# raw_dataset = load_dataset("shahules786/PoetryFoundationData", split="train")

# print(f"Total raw poems available: {len(raw_dataset):,}")
# print("Sample columns:", raw_dataset.column_names)

# # 1. Test the Romanized Hindi dataset
# print("Loading Romanized Hindi Poetry...")
# roman_hindi = load_dataset("ReySajju742/Hindi-Poetry-Dataset", split="train")
# print("Romanized Columns:", roman_hindi.column_names)
# print("Romanized Sample Row 0:", roman_hindi[0])

print("1. Downloading Romanized poetry dataset from Hugging Face...")
dataset = load_dataset("keplersystems/UrduPoetry-35k", split="train")

print(f"Dataset downloaded! Total rows available: {len(dataset):,}")
print(f"raw_poetry_dataset [0] = {dataset[0]}")

Inspect the data

In [ ]:
column_names = dataset.column_names
column_names

Format the data

In [ ]:
import numpy as np

raw_subset = dataset.select(range(1100))
processed_records = []

for row in raw_subset:
  msgs = row['messages']
  user_prompt = msgs[0]['content']
  poem_response = msgs[1]['content']
  normalized_prompt = (
        user_prompt.replace("Roman Urdu", "Hinglish")
                   .replace("ghazal", "poem")
                   .replace("sher", "verse")
    )
  processed_records.append({
        "messages": [
            {"role": "user", "content": normalized_prompt},
            {"role": "assistant", "content": poem_response}
        ]
    })

In [ ]:
from datasets import Dataset
full_dataset = Dataset.from_list(processed_records)

In [ ]:
split_data = full_dataset.train_test_split(test_size=100, seed=42)
train_dataset = split_data['train']
eval_dataset = split_data['test']

print(f"Dataset Split Ready:")
print(f"  Training samples:   {len(train_dataset)}")
print(f"  Validation samples: {len(eval_dataset)}")

In [ ]:
token_lengths = []
for sample in train_dataset:
  formatted_chat = tokenizer.apply_chat_template(sample['messages'], tokenize=False)
  tokens = tokenizer(formatted_chat)['input_ids']
  token_lengths.append(len(tokens))

token_lengths = np.array(token_lengths)
print("=== TOKEN LENGTH AUDIT ===")
print(f"Min Length:        {token_lengths.min()} tokens")
print(f"Average Length:    {token_lengths.mean():.1f} tokens")
print(f"Median (50th %):   {np.percentile(token_lengths, 50):.1f} tokens")
print(f"95th Percentile:   {np.percentile(token_lengths, 95):.1f} tokens")
print(f"Max Length:        {token_lengths.max()} tokens")

In [ ]:
print("=== SAMPLE 0 (RENDERED INPUT) ===\n")
sample_rendered = tokenizer.apply_chat_template(train_dataset[0]["messages"], tokenize=False)
print(sample_rendered)


print("\n=== RAW TOKENS (FIRST 15) ===")
first_few_ids = tokenizer(sample_rendered)["input_ids"][:15]
print("IDs:", first_few_ids)
print("Decoded chunks:", [tokenizer.decode([i]) for i in first_few_ids])

# Train LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                           # Bottleneck rank dimension
    lora_alpha=32,                  # Scaling factor (alpha / r = 2.0)
    lora_dropout=0.05,              # Regularization dropout probability
    target_modules=[                # Apply LoRA to all linear projections
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none"                     # Do not train bias terms
)

In [ ]:
peft_model = get_peft_model(model, peft_config)
print("=== PARAMETER AUDIT ===")
peft_model.print_trainable_parameters()

# Perform SFTT. Supervised Fine-Tune Training

In [ ]:
from trl import SFTTrainer, SFTConfig

In [ ]:
training_args = SFTConfig(
    output_dir="./qwen_hinglish_poetry",
    num_train_epochs=3,                  # 3 passes over the 1,000 poems
    per_device_train_batch_size=2,       # Keeps VRAM footprint tiny
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,       # Effective batch size = 2 * 4 = 8
    learning_rate=2e-4,                  # Standard high LR for LoRA adapters
    lr_scheduler_type="cosine",          # Smooth decay
    warmup_steps=19,                   # 5% warmup steps
    logging_steps=10,                    # Print loss every 10 steps
    eval_strategy="steps",               # Check validation loss periodically
    eval_steps=50,                       # Evaluate on 100 unseen poems every 50 steps
    save_strategy="no",                  # We will save manually at the end
    fp16=True,                           # 16-bit half precision
    max_length=384,                      # Derived from our Phase 2 95th-percentile audit
    packing=False,                       # Keep sequences standard
    report_to="none"                     # Disable external logging like WandB
)

In [ ]:
# 2. Instantiate SFTTrainer
trainer = SFTTrainer(
    model=peft_model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    processing_class=tokenizer,
)

print("SFTTrainer is successfully initialized and ready to train!")

In [ ]:
print("--- LAUNCHING TRAINING RUN ---")
train_result = trainer.train()

print("\n--- TRAINING COMPLETE ---")
print(f"Total Training Time: {train_result.metrics['train_runtime']:.2f} seconds")
print(f"Final Training Loss: {train_result.metrics['train_loss']:.4f}")

In [ ]:
# Save the model

ADAPTER_DIR = "qwen_hinglish_poetry_adapter"
peft_model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to ./{ADAPTER_DIR}")



# To Reload later:

# from peft import PeftModel
# base = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")
# model = PeftModel.from_pretrained(base, ADAPTER_DIR)

In [ ]:
import torch

# Put the model into evaluation mode (disables dropout)
peft_model.eval()

def generate_poetry(prompt_text, temperature=0.7, top_p=0.9):
    messages = [
        {"role": "user", "content": prompt_text}
    ]
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_ids = peft_model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )

    # Strip the input tokens to show only the newly generated verse
    generated_tokens = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

with peft_model.disable_adapter():
    english_poem = generate_poetry("Write a 4-line poem about rain on mountain roads.")

# 2. Full Hinglish (Uses 100% adapter)
hinglish_poem = generate_poetry("Write a short Hinglish poem about safar aur khamoshi.")

print(english_poem)
print(hinglish_poem)

In [ ]:
import time
from transformers import TextIteratorStreamer
from threading import Thread

def benchmark_inference(prompt_text, max_tokens=100):
    messages = [{"role": "user", "content": prompt_text}]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    generate_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_tokens,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id
    )

    # Reset peak memory stats
    torch.cuda.reset_peak_memory_stats()

    # Launch generation in background thread to stream tokens
    thread = Thread(target=peft_model.generate, kwargs=generate_kwargs)

    start_time = time.perf_counter()
    thread.start()

    first_token_time = None
    generated_text = ""
    token_count = 0

    for new_text in streamer:
        if first_token_time is None:
            first_token_time = time.perf_counter()
        generated_text += new_text
        token_count += 1

    thread.join()
    end_time = time.perf_counter()

    # Calculate performance numbers
    ttft_ms = (first_token_time - start_time) * 1000 if first_token_time else 0
    decode_time_s = end_time - first_token_time if first_token_time else (end_time - start_time)
    tps = (token_count - 1) / decode_time_s if decode_time_s > 0 else 0
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

    print("=== INFERENCE BENCHMARK REPORT ===")
    print(f"Prompt: '{prompt_text}'")
    print(f"Tokens Generated:      {token_count}")
    print(f"TTFT (Time to 1st Token): {ttft_ms:.2f} ms")
    print(f"Throughput:            {tps:.2f} tokens/sec")
    print(f"Total Latency:         {(end_time - start_time)*1000:.2f} ms")
    print(f"Peak GPU VRAM Used:    {peak_vram_gb:.2f} GB")
    print("-----------------------------------")
    print(f"Output:\n{generated_text.strip()}\n")

    return {
        "ttft_ms": ttft_ms,
        "tps": tps,
        "vram_gb": peak_vram_gb
    }

# Run the benchmark
bench_results = benchmark_inference("Write a short Hinglish poem about safar aur khamoshi.")

In [ ]:
import matplotlib.pyplot as plt

# Extract training history from trainer state
log_history = trainer.state.log_history

train_steps, train_loss = [], []
eval_steps, eval_loss = [], []

for entry in log_history:
    if "loss" in entry and "step" in entry:
        train_steps.append(entry["step"])
        train_loss.append(entry["loss"])
    if "eval_loss" in entry and "step" in entry:
        eval_steps.append(entry["step"])
        eval_loss.append(entry["eval_loss"])

# Configure plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=300)

# Chart 1: Training vs Validation Loss Curve
ax1.plot(train_steps, train_loss, label="Training Loss", color="#2563EB", lw=2.2)
if eval_steps:
    ax1.plot(eval_steps, eval_loss, label="Validation Loss (Eval)", color="#DC2626", lw=2.2, marker="o", markersize=5)
ax1.set_title("Cross-Entropy Loss Progression across 3 Epochs", fontsize=12, fontweight='bold', pad=10)
ax1.set_xlabel("Optimization Steps (Batch Size = 8)", fontsize=10)
ax1.set_ylabel("Loss", fontsize=10)
ax1.legend(frameon=True, facecolor='white', framealpha=0.9)
ax1.grid(True, linestyle="--", alpha=0.6)

# Chart 2: Trainable Parameter Efficiency Breakdown
param_labels = ["Base Frozen\n(494M)", "LoRA Adapter\n(8.8M)"]
param_values = [494.0, 8.8]
colors = ["#94A3B8", "#10B981"]

bars = ax2.bar(param_labels, param_values, color=colors, width=0.45)
ax2.set_yscale("log")
ax2.set_title("Parameter Footprint: 98.25% Reduction", fontsize=12, fontweight='bold', pad=10)
ax2.set_ylabel("Parameters (Millions - Log Scale)", fontsize=10)
for bar in bars:
    yval = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2.0, yval * 1.15, f"{yval:.1f}M", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig("finetuning_metrics.png", dpi=300)
plt.show()

print("Chart saved successfully as 'finetuning_metrics.png'!")